In [1]:
import pandas as pd
import numpy as np
import xarray as xr
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, Dataset, DataLoader, random_split
import matplotlib.pyplot as plt
import os

from models import FluxNet
from models import compute_divergence_field

device = "cuda" if torch.cuda.is_available() else "cpu"

# RETRAIN = False
RETRAIN = True

In [2]:
data = torch.load("data/flux_tensor.pt", weights_only = False)

In [3]:
# Initialise
model = FluxNet().to(device)
# Load trained weights
model.load_state_dict(torch.load("trained_models/fluxnet_all_data.pth", map_location = device))
model.eval()

/tmp/ipykernel_237896/2746205672.py:4: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("trained_models/fluxnet_all_data.pth", map_location = d

FluxNet(
  (inp): Sequential(
    (0): Linear(in_features=2, out_features=256, bias=True)
    (1): SiLU()
  )
  (trunk): ModuleList(
    (0-5): 6 x _ResBlock(
      (lin1): Linear(in_features=256, out_features=256, bias=True)
      (lin2): Linear(in_features=256, out_features=256, bias=True)
      (act): SiLU()
    )
  )
  (head_df_psi): Linear(in_features=256, out_features=1, bias=True)
  (head_cf_phi): Linear(in_features=256, out_features=1, bias=True)
)

In [4]:
from regions import ROSS_BOUNDS

x_min = ROSS_BOUNDS["x_min"]
x_max = ROSS_BOUNDS["x_max"]
y_min = ROSS_BOUNDS["y_min"]
y_max = ROSS_BOUNDS["y_max"]

In [5]:
target_grid_mask = xr.load_dataset("data/target_grid_mask.nc")

X, Y = xr.broadcast(target_grid_mask.x, target_grid_mask.y)

X_long = X.values.flatten()
Y_long = Y.values.flatten()

# NOTE: Normalise to [0, 1] as model input
X_long = (X_long - x_min) / (x_max - x_min)
Y_long = (Y_long - y_min) / (y_max - y_min)

# mask_long = target_grid_mask.mask.values.flatten()

XY_long_tensor = torch.cat((torch.tensor(X_long).unsqueeze(1), torch.tensor(Y_long).unsqueeze(1)), dim = 1)
print(XY_long_tensor.shape)
print(XY_long_tensor[0:5, :])

torch.Size([4000000, 2])
tensor([[2.5000e-04, 9.9975e-01],
        [2.5000e-04, 9.9925e-01],
        [2.5000e-04, 9.9875e-01],
        [2.5000e-04, 9.9825e-01],
        [2.5000e-04, 9.9775e-01]])


In [7]:
# bounds
x0, x1 = 0.25, 0.30
y0, y1 = 0.25, 0.30

# boolean mask for the subset (works for numpy arrays or torch tensors)
sel = (X_long >= x0) & (X_long <= x1) & (Y_long >= y0) & (Y_long <= y1)

# subset arrays
X_sub = X_long[sel]
Y_sub = Y_long[sel]

# build tensor on the subset
XY_sub_tensor = torch.cat(
    (torch.tensor(X_sub).unsqueeze(1), torch.tensor(Y_sub).unsqueeze(1)),
    dim=1
)

print(XY_sub_tensor.shape)
print(XY_sub_tensor[:5, :])

torch.Size([10000, 2])
tensor([[0.2503, 0.2998],
        [0.2503, 0.2993],
        [0.2503, 0.2988],
        [0.2503, 0.2982],
        [0.2503, 0.2977]])


In [ ]:
XY_batch = XY_sub_tensor.requires_grad_(True).to(device)

flux_predictions_batch, df, cf = model(XY_batch, return_parts = True)

div_batch = compute_divergence_field(
    mean_pred = flux_predictions_batch, 
    x_grad = XY_batch)

In [11]:
import numpy as np
import torch
import matplotlib.pyplot as plt

# --- run model on subset ---
XY_batch = XY_sub_tensor.requires_grad_(True).to(device)
flux_predictions_batch, df_batch, cf_batch = model(XY_batch, return_parts=True)

div_batch = compute_divergence_field(
    mean_pred=flux_predictions_batch,
    x_grad=XY_batch
)

# --- helper: reshape scattered (x,y)->grid and plot quiver ---
def quiver_on_grid(ax, XY_cpu, V_cpu, title, stride=2):
    """
    XY_cpu: (N,2) numpy, V_cpu: (N,2) numpy
    Assumes XY points lie on a rectilinear grid (unique x and unique y).
    """
    x = XY_cpu[:, 0]
    y = XY_cpu[:, 1]
    vx = V_cpu[:, 0]
    vy = V_cpu[:, 1]

    xu = np.unique(x)
    yu = np.unique(y)
    Nx, Ny = xu.size, yu.size

    # map each (x,y) to grid index
    ix = np.searchsorted(xu, x)
    iy = np.searchsorted(yu, y)

    VX = np.full((Ny, Nx), np.nan, dtype=float)
    VY = np.full((Ny, Nx), np.nan, dtype=float)

    VX[iy, ix] = vx
    VY[iy, ix] = vy

    Xg, Yg = np.meshgrid(xu, yu)

    # thin out arrows for readability
    ax.quiver(
        Xg[::stride, ::stride],
        Yg[::stride, ::stride],
        VX[::stride, ::stride],
        VY[::stride, ::stride],
    )
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.set_xlabel("x")
    ax.set_ylabel("y")

# --- move to CPU numpy ---
XY_cpu   = XY_batch.detach().cpu().numpy()
F_cpu    = flux_predictions_batch.detach().cpu().numpy()
DF_cpu   = df_batch.detach().cpu().numpy()
CF_cpu   = cf_batch.detach().cpu().numpy()
DIV_cpu  = div_batch.detach().cpu().numpy()  # shape (N,) or (N,1)

# --- plot three vector fields ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)

quiver_on_grid(axes[0], XY_cpu, F_cpu,  "Flux (total)", stride=2)
quiver_on_grid(axes[1], XY_cpu, DF_cpu, "Divergence-free part (df)", stride=2)
quiver_on_grid(axes[2], XY_cpu, CF_cpu, "Curl-free part (cf)", stride=2)

plt.show()

# --- optional: also visualise divergence as a scalar field on the same grid ---
DIV_cpu = DIV_cpu.squeeze()

xu = np.unique(XY_cpu[:, 0])
yu = np.unique(XY_cpu[:, 1])
ix = np.searchsorted(xu, XY_cpu[:, 0])
iy = np.searchsorted(yu, XY_cpu[:, 1])

DIV_grid = np.full((yu.size, xu.size), np.nan, dtype=float)
DIV_grid[iy, ix] = DIV_cpu

Xg, Yg = np.meshgrid(xu, yu)

fig, ax = plt.subplots(figsize=(6, 5), constrained_layout=True)
pcm = ax.pcolormesh(Xg, Yg, DIV_grid, shading="auto", cmap="RdBu")
ax.set_title("Divergence of flux (∇·v)")
ax.set_aspect("equal")
fig.colorbar(pcm, ax=ax)
plt.show()


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 23.64 GiB of which 192.19 MiB is free. Process 232530 has 11.60 GiB memory in use. Including non-PyTorch memory, this process has 11.67 GiB memory in use. Of the allocated memory 11.21 GiB is allocated by PyTorch, and 5.93 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)